# proj08 Chameleon KVM Provisioning Notebook

This notebook is the supported way to reserve and create the three Chameleon `m1.large` VMs for the proj08 Smart Transaction Categorization Kubernetes demo.

## What actually worked last time

The successful path was not plain Terraform. Terraform repeatedly hit Chameleon/OpenStack authentication and service-catalog issues from the Jupyter shell environment:

- short-lived notebook `OS_ACCESS_TOKEN` variables pointed at the wrong site or expired;
- `regionOne` catalog discovery worked for token issue, but compute/network operations needed `KVM@TACC`;
- Terraform's OpenStack provider could not reliably resolve the KVM networking endpoint in that mixed environment.

The working flow was:

1. Reserve three `m1.large` VMs with the Chameleon `chi` Python lease API.
2. Use the lease's reserved flavor id as the OpenStack `--flavor` value.
3. Create a KVM@TACC application credential in Horizon and save it to `~/.config/openstack/clouds.yaml`.
4. Run `scripts/create-chameleon-kvm-cluster.sh`, which wraps the tested OpenStack CLI provisioning path.

The April 2026 successful run used this shape:

```bash
KEY_NAME=id_rsa_chameleon bash scripts/create-chameleon-kvm-cluster.sh 3bb9f2d9-6dd7-4d87-ba7b-74cfb437af8b
```

That script created:

- private network `private-net-mlops-proj08` with nodes `192.168.1.11/12/13`;
- a public security group opening the project demo ports;
- three Ubuntu 24.04 VMs named `node1-mlops-proj08`, `node2-mlops-proj08`, `node3-mlops-proj08`;
- a floating IP on node1;
- updated Ansible inventory files for Kubespray.


In [ ]:
from chi import context, lease
import chi
import datetime
from pathlib import Path

PROJECT_SUFFIX = "proj08"
LEASE_NAME = "k8s-cluster-zy3180"
NODE_COUNT = 3
LEASE_HOURS = 168  # one week for the production-style traffic window
FLAVOR_NAME = "m1.large"
KEY_NAME = "id_rsa_chameleon"
REPO_DIR = Path("/work/proj08-iac")
RESERVATION_ENV = REPO_DIR / ".proj08-reservation.env"

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

print({
    "project_suffix": PROJECT_SUFFIX,
    "lease_name": LEASE_NAME,
    "node_count": NODE_COUNT,
    "flavor": FLAVOR_NAME,
    "lease_hours": LEASE_HOURS,
    "repo_dir": str(REPO_DIR),
})


## 1. Clone or refresh the DevOps repo in Chameleon Jupyter

Run this cell in the Chameleon Jupyter terminal/notebook environment. It keeps `/work/proj08-iac` on the latest `main` branch before provisioning.


In [ ]:
%%bash
set -euo pipefail
cd /work
if [ ! -d proj08-iac/.git ]; then
  git clone https://github.com/ch4r1ty/proj08-iac.git
fi
cd proj08-iac
git pull origin main
bash scripts/install-terraform.sh || true
export PATH="$HOME/.local/bin:/work/.local/bin:$PATH"
git log --oneline -3


## 2. Reserve three `m1.large` VMs

This creates or reuses the lease. Chameleon exposes the reserved capacity as a reserved flavor id. That id is what the OpenStack CLI script passes to `openstack server create --flavor`.


In [ ]:
l = lease.Lease(LEASE_NAME, duration=datetime.timedelta(hours=LEASE_HOURS))
l.add_flavor_reservation(id=chi.server.get_flavor_id(FLAVOR_NAME), amount=NODE_COUNT)
l.submit(idempotent=True)
l.show()

reserved_flavor = l.get_reserved_flavors()[0]
RESERVED_FLAVOR_ID = reserved_flavor.id
RESERVED_FLAVOR_NAME = reserved_flavor.name

print("Reserved flavor name:", RESERVED_FLAVOR_NAME)
print("Reserved flavor id:  ", RESERVED_FLAVOR_ID)

REPO_DIR.mkdir(parents=True, exist_ok=True)
RESERVATION_ENV.write_text(
    f"PROJECT_SUFFIX={PROJECT_SUFFIX}\n"
    f"LEASE_NAME={LEASE_NAME}\n"
    f"RESERVED_FLAVOR_ID={RESERVED_FLAVOR_ID}\n"
    f"RESERVED_FLAVOR_NAME={RESERVED_FLAVOR_NAME}\n"
    f"KEY_NAME={KEY_NAME}\n"
)
print("Wrote", RESERVATION_ENV)


## 3. Create KVM@TACC OpenStack credentials

The provisioning script intentionally ignores notebook-provided `OS_*` variables and reads `~/.config/openstack/clouds.yaml` instead. This avoids the wrong-site/expired-token problem.

Create a KVM@TACC Application Credential in Horizon:

1. Open KVM@TACC Horizon.
2. Go to **Identity -> Application Credentials**.
3. Create a credential that lasts beyond the demo window.
4. Copy the generated id and secret.
5. Run the helper below in a terminal, then paste the id and secret when prompted.

The secret is never stored in the notebook output.


In [ ]:
%%bash
set -euo pipefail
cd /work/proj08-iac
cat <<'MSG'
If ~/.config/openstack/clouds.yaml does not exist yet, run this in a terminal:

  cd /work/proj08-iac
  bash scripts/configure-kvm-clouds-yaml.sh

Then continue with the next cell.
MSG

if [ -f "$HOME/.config/openstack/clouds.yaml" ]; then
  bash scripts/fix-kvm-cloud-region.sh
  bash scripts/openstack-kvm.sh token issue -f table
  bash scripts/openstack-kvm.sh keypair list
else
  echo "clouds.yaml is missing; create it before running the provisioning cell."
fi


## 4. Create the three VM servers

This is the combined program. It uses the reserved flavor id from the lease cell and creates the full VM layer:

- private network and subnet;
- public service security group;
- node1/node2/node3 sharednet and private ports;
- three Ubuntu VMs;
- node1 floating IP;
- Ansible/Kubespray inventory updates;
- `artifacts/chameleon/latest-cluster.env` with the result.


In [ ]:
%%bash
set -euo pipefail
cd /work/proj08-iac
source .proj08-reservation.env
KEY_NAME="${KEY_NAME:-id_rsa_chameleon}" \
SUFFIX="${PROJECT_SUFFIX:-proj08}" \
bash scripts/create-chameleon-kvm-cluster.sh "${RESERVED_FLAVOR_ID}"


## 5. Read the result and show next commands

After this cell prints a floating IP, SSH to node1 and continue with Kubespray + stack deployment.


In [ ]:
from pathlib import Path

env_file = Path("/work/proj08-iac/artifacts/chameleon/latest-cluster.env")
values = {}
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            key, value = line.split("=", 1)
            values[key] = value.strip().strip('"')

floating_ip = values.get("FLOATING_IP", "<FLOATING_IP>")
print("Floating IP:", floating_ip)
print()
print("SSH:")
print(f"ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
print()
print("After Kubespray installs Kubernetes on node1:")
print("cd /home/cc/proj08-iac")
print("export KUBECONFIG=/etc/kubernetes/admin.conf")
print(f"bash scripts/deploy-k8s-stack.sh {floating_ip}")


## 6. Service URL checklist after deployment

Use Actual HTTPS for the UI. Public HTTP is only a health/debug fallback because Chrome blocks `SharedArrayBuffer` on non-secure public origins.


In [ ]:
FLOATING_IP = values.get("FLOATING_IP", "<FLOATING_IP>") if "values" in globals() else "<FLOATING_IP>"
service_urls = {
    "Actual Budget HTTPS": f"https://{FLOATING_IP}:30443",
    "Actual Budget HTTPS sslip": f"https://actual.{FLOATING_IP}.sslip.io:30443",
    "Actual Budget HTTP health": f"http://{FLOATING_IP}:30083/health",
    "SmartCat Serving docs": f"http://{FLOATING_IP}:30090/docs",
    "Grafana": f"http://{FLOATING_IP}:30300",
    "Prometheus": f"http://{FLOATING_IP}:30909",
    "MLflow": f"http://{FLOATING_IP}:8000",
    "MinIO console": f"http://{FLOATING_IP}:9001",
}
for name, url in service_urls.items():
    print(f"{name:30s} {url}")

print("\nCredentials:")
print("Grafana: admin / admin123")
print("MinIO:   your-access-key / PL89sTsyClxjtvQVxjN9")
print("MLflow, Prometheus, SmartCat docs: no login")


## 7. Import Grafana dashboards

After `scripts/deploy-k8s-stack.sh` finishes and Grafana is reachable, run this cell to import the standard monitoring dashboards used by the demo:

- `1860`: Node Exporter Full
- `3119`: Kubernetes cluster monitoring
- `15760`: Kubernetes Views Pods

The script uses Grafana API credentials `admin/admin123` and maps all Prometheus datasource inputs to datasource UID `prometheus`.


In [ ]:
from pathlib import Path
import os
import subprocess

if "values" in globals():
    FLOATING_IP = values.get("FLOATING_IP", "<FLOATING_IP>")
else:
    FLOATING_IP = globals().get("FLOATING_IP", "<FLOATING_IP>")

if FLOATING_IP == "<FLOATING_IP>":
    raise RuntimeError("Set FLOATING_IP first, or run the provisioning result cell above.")

env = os.environ.copy()
env.update({
    "GRAFANA_URL": f"http://{FLOATING_IP}:30300",
    "GRAFANA_USER": "admin",
    "GRAFANA_PASSWORD": "admin123",
    "PROMETHEUS_UID": "prometheus",
    "DASHBOARD_IDS": "1860 3119 15760",
})

subprocess.run(
    ["bash", "scripts/import-grafana-dashboards.sh"],
    cwd=Path("/work/proj08-iac"),
    env=env,
    check=True,
)
print(f"Open Grafana dashboards: http://{FLOATING_IP}:30300/dashboards")


## 8. Quick health checks from your laptop

After deployment finishes, these should return HTTP 200:

```bash
curl -k -I https://<FLOATING_IP>:30443
curl http://<FLOATING_IP>:30090/healthz
curl http://<FLOATING_IP>:30909/-/ready
curl http://<FLOATING_IP>:8000
curl http://<FLOATING_IP>:9001
```

For demo traffic against SmartCat:

```bash
curl -s -X POST http://<FLOATING_IP>:30090/predict \
  -H 'Content-Type: application/json' \
  -d '{"transaction_description":"Starbucks coffee lunch","amount":8.25,"currency":"USD","country":"US"}' | jq
```
